In [29]:
# pip install scikit-image

Yang dilakukan: 
1. Mengubah seluruh resolusi data dari satelit yang berbeda beda menjadi satu standar ukuran grid
2. Membuat grid spasial wilayah Kalimantan dengan resolusi 0.25° x 0.25°
3. Memastikan setiap piksel data suhu, hujan, dan vegetasi "menumpuk" di lokasi koordinat (Latitude/Longitude) yang tepat

In [30]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime, timedelta
from netCDF4 import Dataset
from skimage.transform import resize

In [ ]:
MIN_LON, MAX_LON = 108.0, 119.5
MIN_LAT, MAX_LAT = -5.0, 5.0

GRID_RES = 0.25 
WINDOW_SIZE = 7

INPUT_CSV = "../data/processed/dataset_FireFinder_ai_kalimantan.csv"
RAW_LST_DIR = "../data/raw/lst"
RAW_RAIN_DIR = "../data/raw/rainfall"
RAW_VEG_DIR = "../data/raw/vegetation"
SAVE_FOLDER = "../data/processed"

lat_range = np.arange(MIN_LAT, MAX_LAT, GRID_RES)
lon_range = np.arange(MIN_LON, MAX_LON, GRID_RES)
n_lat = len(lat_range)
n_lon = len(lon_range)

print(f"⚙️ GRID CONFIGURATION:")
print(f"   - Target Grid Size: {n_lat} (Height) x {n_lon} (Width)")

⚙️ GRID CONFIGURATION:
   - Target Grid Size: 40 (Height) x 46 (Width)


In [ ]:

def read_and_resize_lst(date_obj, target_shape=(n_lat, n_lon)):
    """Membaca LST Raw -> Resize ke Grid"""
    try:
        julian_str = f"A{date_obj.year}{date_obj.timetuple().tm_yday:03d}"
        search_pattern = os.path.join(RAW_LST_DIR, f"*{julian_str}*.hdf")
        found_files = glob.glob(search_pattern)
        
        if not found_files: return np.zeros(target_shape)
            
        with Dataset(found_files[0], mode='r') as ds:

            var_name = [k for k in ds.variables.keys() if "LST_Day" in k][0]
            data = ds.variables[var_name][:]
            data = data.astype(float)
            data[data < 200] = np.nan 
            data = data * 0.02 - 273.15 
            data = np.nan_to_num(data, nan=30.0) 
            
            return resize(data, target_shape, mode='reflect', anti_aliasing=True)
            
    except Exception:
        return np.zeros(target_shape)

def read_and_resize_rain(date_obj, target_shape=(n_lat, n_lon)):
    """Membaca GPM Rain Raw -> Resize ke Grid"""
    try:
        date_str = date_obj.strftime("%Y-%m-%d")
        fname = os.path.join(RAW_RAIN_DIR, f"gpm_{date_str}.nc4")
        
        if not os.path.exists(fname): return np.zeros(target_shape)
            
        with Dataset(fname, mode='r') as ds:
            grp = ds.groups['Grid'] if 'Grid' in ds.groups else ds
            precip = grp.variables['precipitation'][:] 
            
            raw_rain = precip[0].T 
            raw_rain[raw_rain < 0] = 0
            
            return resize(raw_rain, target_shape, mode='reflect', anti_aliasing=True)

    except Exception:
        return np.zeros(target_shape)

ndvi_cache = {} 
def read_and_resize_ndvi(date_obj, target_shape=(n_lat, n_lon)):
    """Membaca NDVI Raw -> Resize ke Grid"""
    date_str = date_obj.strftime("%Y-%m-%d")
    if date_str in ndvi_cache: return ndvi_cache[date_str]
    
    try:
        julian_year = f"A{date_obj.year}" 
        search_pattern = os.path.join(RAW_VEG_DIR, f"*{julian_year}*.hdf")
        files = sorted(glob.glob(search_pattern))
        
        if not files: return np.full(target_shape, 0.6)
            
        target_file = files[-1] 
        with Dataset(target_file, mode='r') as ds:
            ds.set_auto_maskandscale(False)
            var_name = [k for k in ds.variables.keys() if "NDVI" in k][0]
            data = ds.variables[var_name][:]
            data = data.astype(float) * 0.0001
            data[data < -0.2] = 0
            
            resized = resize(data, target_shape, mode='reflect', anti_aliasing=True)
            ndvi_cache[date_str] = resized
            return resized
            
    except Exception:
        return np.full(target_shape, 0.6)

In [ ]:
print("🚀 Memulai Generasi Grid Spasial...")

df = pd.read_csv(INPUT_CSV)
initial_len = len(df)

def clean_confidence(val):
    try:
        if isinstance(val, str):
            val = val.lower().strip()
            if val == 'n': return 50
            if val == 'l': return 10
            if val == 'h': return 90
            return 0
        return float(val)
    except:
        return 0

df['confidence_score'] = df['confidence'].apply(clean_confidence)

df = df[df['confidence_score'] >= 30]

if 'type' in df.columns:
    df = df[df['type'] == 0]

print(f"🧹 Data Cleaning: {initial_len} -> {len(df)} titik api valid.")

df['acq_date'] = pd.to_datetime(df['acq_date'])

start_date = df['acq_date'].min()
end_date = df['acq_date'].max()
all_dates = pd.date_range(start=start_date, end=end_date)

print(f" -> Rentang Waktu: {start_date.date()} s/d {end_date.date()} ({len(all_dates)} hari)")

daily_grids_list = []

for idx, d in enumerate(all_dates):
    if idx % 10 == 0: print(f"   Processing hari ke-{idx}: {d.date()}...")
    
    df_day = df[df['acq_date'] == d]
    if df_day.empty:
        hotspot_mat = np.zeros((n_lat, n_lon))
    else:
        hotspot_mat, _, _ = np.histogram2d(
            df_day['latitude'], df_day['longitude'], 
            bins=[n_lat, n_lon], 
            range=[[MIN_LAT, MAX_LAT], [MIN_LON, MAX_LON]]
        )
        hotspot_mat = np.clip(hotspot_mat, 0, 1)

    lst_layer = read_and_resize_lst(d, (n_lat, n_lon))
    rain_layer = read_and_resize_rain(d, (n_lat, n_lon))
    ndvi_layer = read_and_resize_ndvi(d, (n_lat, n_lon))
    
    daily_grid = np.stack([hotspot_mat, lst_layer, rain_layer, ndvi_layer], axis=-1)
    daily_grids_list.append(daily_grid)

daily_grids = np.array(daily_grids_list)
print(f"✅ Grid Generation Selesai. Shape Sementara: {daily_grids.shape}")

🚀 Memulai Generasi Grid Spasial...
🧹 Data Cleaning: 200906 -> 166359 titik api valid.
 -> Rentang Waktu: 2023-01-01 s/d 2023-12-31 (365 hari)
   Processing hari ke-0: 2023-01-01...
   Processing hari ke-10: 2023-01-11...
   Processing hari ke-20: 2023-01-21...
   Processing hari ke-30: 2023-01-31...
   Processing hari ke-40: 2023-02-10...
   Processing hari ke-50: 2023-02-20...
   Processing hari ke-60: 2023-03-02...
   Processing hari ke-70: 2023-03-12...
   Processing hari ke-80: 2023-03-22...
   Processing hari ke-90: 2023-04-01...
   Processing hari ke-100: 2023-04-11...
   Processing hari ke-110: 2023-04-21...
   Processing hari ke-120: 2023-05-01...
   Processing hari ke-130: 2023-05-11...
   Processing hari ke-140: 2023-05-21...
   Processing hari ke-150: 2023-05-31...
   Processing hari ke-160: 2023-06-10...
   Processing hari ke-170: 2023-06-20...
   Processing hari ke-180: 2023-06-30...
   Processing hari ke-190: 2023-07-10...
   Processing hari ke-200: 2023-07-20...
   Proce

In [34]:
# ================= 4. SLIDING WINDOW & SAVE =================
print(f" -> Membuat Sequence Window ({WINDOW_SIZE} hari)...")

X_seq, Y_seq = [], []

# Pastikan data cukup untuk sliding window
if len(daily_grids) > WINDOW_SIZE:
    for i in range(len(daily_grids) - WINDOW_SIZE):
        # Input: 7 Hari ke belakang
        x_window = daily_grids[i : i + WINDOW_SIZE]
        
        # Target: Hari ke-8 (Hanya peta api - Channel 0)
        target_grid = daily_grids[i + WINDOW_SIZE, :, :, 0] 
        target_grid = np.expand_dims(target_grid, axis=-1)
        
        X_seq.append(x_window)
        Y_seq.append(target_grid)

    X_train = np.array(X_seq)
    y_train = np.array(Y_seq)

    os.makedirs(SAVE_FOLDER, exist_ok=True)
    np.save(os.path.join(SAVE_FOLDER, "X_TRAIN_GRID.npy"), X_train)
    np.save(os.path.join(SAVE_FOLDER, "Y_TRAIN_GRID.npy"), y_train)

    print(f"\n✅ SUKSES FINAL! Dataset Spasial tersimpan.")
    print(f"   X Shape: {X_train.shape}")
    print(f"   Y Shape: {y_train.shape}")
else:
    print("❌ Error: Jumlah hari tidak cukup untuk membuat window sequence.")

 -> Membuat Sequence Window (7 hari)...

✅ SUKSES FINAL! Dataset Spasial tersimpan.
   X Shape: (358, 7, 40, 46, 4)
   Y Shape: (358, 40, 46, 1)
